In [2]:
# This notebook uses the original cell sequence; source files are read from Datasets/.
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'Project_machine_notebook.ipynb').is_file(): PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT / 'Datasets')


# Phase 2: Machine Learning Implementation
## Multi-Target Football Match Performance Predictor
**Track:** Sports & Fitness | **Course:** CSCI417 Machine Intelligence  
**Model Assigned:** Random Forest Classifier (Match Outcome Prediction)  

---
**Objective:** To train, evaluate, and optimize models to predict the outcome of a football match (Win, Draw, Loss) based purely on in-game performance metrics.

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv("final_verified_dataset.csv")

df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")
df = df.sort_values(["Team", "date"]).reset_index(drop=True)

In [5]:
df.columns

Index(['League', 'season', 'Corner kicks', 'Expected goals (xG)',
       'Ball possession', 'Shots off target', 'Total shots', 'Shots on target',
       'Yellow cards', 'Red cards', 'Fouls', 'Team', 'Opponent', 'Goals for',
       'Goals against', 'venue', 'result', 'Total shots_sqrt',
       'Expected goals (xG)_log', 'Goals for_log', 'Goals against_log',
       'date'],
      dtype='str')

In [6]:
STAT_COLS = [
    "Corner kicks",
    "Expected goals (xG)",
    "Ball possession",
    "Shots off target",
    "Total shots",
    "Shots on target",
    "Yellow cards",
    "Red cards",
    "Fouls",
    "Goals for",
    "Goals against",
]
N_SPLITS  = 5
EARLY_STOP = 75
SEED       = 42
WINDOWS    = [3, 5, 10]

LABEL_NAMES = {0: "Loss", 1: "Draw", 2: "Win"}

### Derived Match Features
Create target labels, home/away indicators, and single-match derived statistics for each fixture.

In [7]:
# Encode result
df["target"] = df["result"].map({"W":2, "D":1, "L":0})
df["points"] = df["result"].map({"W":3, "D":1, "L":0})
df["is_home"] = (df["venue"].str.strip().str.lower() == "home").astype(int)

# Force numeric stat columns
for col in STAT_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace("%","",regex=False).str.strip().replace("nan",np.nan),
            errors="coerce"
        )

# Derived single‑match columns
df["goal_diff"]       = df["Goals for"] - df["Goals against"]
df["clean_sheet"]     = (df["Goals against"] == 0).astype(float)
df["win_flag"]        = (df["result"] == "W").astype(float)
df["draw_flag"]       = (df["result"] == "D").astype(float)
df["loss_flag"]       = (df["result"] == "L").astype(float)
df["xG_diff"]         = df["Expected goals (xG)"] - df["Goals against"]
df["conversion_rate"] = df["Goals for"] / df["Total shots"].replace(0, np.nan)
df["shots_ratio"]     = df["Shots on target"] / df["Total shots"].replace(0, np.nan)
df["days_rest"] = df.groupby("Team")["date"].transform(lambda s: s.diff().dt.days)

# Drop rows with missing target
df = df.dropna(subset=["date", "target"])


### Rolling Feature Definitions
Define a helper to compute shifted rolling statistics so the model never sees the current match when building features.

In [8]:
# shift(1) then rolling to prevent seeing the current match
def _roll(series, window, func="mean"):
    return getattr(series.shift(1).rolling(window, min_periods=1), func)()
for col in STAT_COLS:
    if col not in df.columns: continue
    safe = col.replace(" ","_").replace("(","").replace(")","")
    for w in WINDOWS:
        df[f"roll{w}_{safe}"] = df.groupby("Team")[col].transform(
            lambda s, w=w: _roll(s, w)
        )



### Form and Streak Features
Create aggregated team form stats, streak labels, and opponent-level defense features from historical matches.

In [9]:
form_map = {
    "points":"points", "goal_diff":"goal_diff", "clean_sheet":"clean_sheet",
    "win_rate":"win_flag", "draw_rate":"draw_flag", "loss_rate":"loss_flag",
    "xG_diff":"xG_diff", "conversion_rate":"conversion_rate", "shots_ratio":"shots_ratio"
}
for fname, src in form_map.items():
    if src not in df.columns: continue
    for w in WINDOWS:
        df[f"roll{w}_{fname}"] = df.groupby("Team")[src].transform(
            lambda s, w=w: _roll(s, w)
        )
def calc_streak(s):
    out, count = [], 0
    for val in s:
        if val == 2:   count = count+1 if count>0 else 1
        elif val == 0: count = count-1 if count<0 else -1
        else:          count = 0
        out.append(count)
    return pd.Series(out, index=s.index)

df["streak_raw"] = df.groupby("Team")["target"].transform(calc_streak)
df["streak"]     = df.groupby("Team")["streak_raw"].transform(lambda s: s.shift(1))
df["days_rest"] = df["days_rest"].fillna(7)

# Opponent rolling goals against / xG allowed
opp_stats = df[["Team","date","Goals against","Expected goals (xG)"]].copy()
opp_stats["opp_def_r5"]    = df.groupby("Team")["Goals against"].transform(lambda s: _roll(s,5))
opp_stats["opp_xg_att_r5"] = df.groupby("Team")["Expected goals (xG)"].transform(lambda s: _roll(s,5))

opp_lookup = opp_stats[["Team","date","opp_def_r5","opp_xg_att_r5"]].rename(columns={"Team":"Opponent"})
df = df.merge(opp_lookup, on=["Opponent","date"], how="left")


### Venue-Specific Rolling Features
Create home/away rolling averages for goals and points using only prior matches.

In [10]:
for venue_val, label in [(1,"home"), (0,"away")]:
    mask = (df["is_home"] == venue_val)
    for src, fname in [("Goals for","gf"), ("Goals against","ga"), ("points","pts")]:
        if src not in df.columns: continue
        new = f"roll5_{fname}_{label}"
        df[new] = df.groupby("Team")[src].transform(
            lambda s: s.where(mask).shift(1).rolling(5, min_periods=1).mean()
        )

### Feature Collection and Imputation
Gather the generated rolling statistics and fill any remaining missing values with medians.

In [11]:
# Collect every feature column we created
feat_cols = [c for c in df.columns if c.startswith("roll")]  # all rolling stats
feat_cols += ["streak","days_rest","opp_def_r5","opp_xg_att_r5","is_home"]
feat_cols = [c for c in feat_cols if c in df.columns]
print(f"[MATRIX] Shape={df.shape}  features={len(feat_cols)}")

# Median‑fill remaining NaNs
medians = df[feat_cols].median()
df[feat_cols] = df[feat_cols].fillna(medians)

print(f"[MATRIX] Shape={df.shape}  features={len(feat_cols)}")

[MATRIX] Shape=(23838, 104)  features=71
[MATRIX] Shape=(23838, 104)  features=71


In [12]:
df.to_csv("final_featured_dataset.csv", index=False)